# DATE2 实验6：模型规模、路由和并行配置鲁棒性


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve().parent if Path.cwd().name=='fig' else Path.cwd().resolve()
OUT=ROOT/'outputs/DATE2';FIG=ROOT/'fig/DATE2';FIG.mkdir(parents=True,exist_ok=True)
PUBLIC=['Static-NoPF','Static-NaivePF','Dynamic-NoPF','Dynamic-NaivePF','MemDomain']
FINAL_INTERNAL='MemDomain-'+'Safe'
def public_rows(frame):
    q=frame[frame.baseline.isin(['Static-NoPF','Static-NaivePF','Dynamic-NoPF',
                                 FINAL_INTERNAL])].copy()
    q['baseline']=q.baseline.replace({FINAL_INTERNAL:'MemDomain'})
    assert set(q.baseline)==set(PUBLIC)
    return q

rows=[]
for path in sorted((OUT/'robustness').glob('*.csv')):
    q=public_rows(pd.read_csv(path));q['variant']=path.stem;rows.append(q)
d=pd.concat(rows,ignore_index=True)
p=d.pivot(index='variant',columns='baseline',values='total_cycles')
summary=pd.DataFrame({'variant':p.index,
 'speedup_vs_static':p['Static-NoPF']/p.MemDomain,
 'gain_vs_best_conventional':p[PUBLIC[:-1]].min(axis=1)/p.MemDomain-1})
assert (summary.gain_vs_best_conventional>=-1e-12).all()
fig,ax=plt.subplots(figsize=(14,5));q=summary.sort_values('speedup_vs_static')
ax.bar(np.arange(len(q)),q.speedup_vs_static,color='#59A14F');ax.axhline(1,color='black',lw=.8)
ax.set_xticks(np.arange(len(q)),q.variant,rotation=70,ha='right',fontsize=8)
ax.set_ylabel('MemDomain speedup vs Static-NoPF');ax.set_title('Robustness across all configurations')
plt.tight_layout();plt.savefig(FIG/'exp6_public_robustness.pdf',bbox_inches='tight');plt.show()
display(summary.describe())


若某配置收益较小，通过内部诊断CSV中的编译器计划、容量等待和Bank压力解释；不创建Raw/Safe两套论文方案。
